In [1]:
import pandas as pd
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL
from dotenv import load_dotenv
import os

# 1. Carregar as variáveis do arquivo .env
load_dotenv()

host = os.getenv("DB_HOST")
port = os.getenv("DB_PORT")
dbname = os.getenv("DB_NAME")
user = os.getenv("DB_USER")
password = os.getenv("DB_PASSWORD")

try:
    # Configurar a conexão com o PostgreSQL
    url = URL.create(
        drivername="postgresql+psycopg",
        username=user,
        password=password,
        host=host,
        port=int(port) if port else 5432,
        database=dbname
    )

    engine = create_engine(url)

    # Consulta SQL
    query = """
        SELECT
            ped.id_pedido,
            prod.categoria,
            ip.quantidade,
            ip.preco_unitario
        FROM pedidos ped
        INNER JOIN itens_pedido ip
            ON ped.id_pedido = ip.id_pedido
        INNER JOIN produtos prod
            ON ip.id_produto = prod.id_produto
        WHERE ped.status = 'pago'
    """

    # Executar a consulta e criar o DataFrame
    with engine.connect() as conectar:
        df_vendas = pd.read_sql(text(query), conectar)

    print("Conexão com PostgreSQL estabelecida!")
    print("Dados carregados com sucesso!")

except Exception as e:
    print("Erro ao carregar os dados de vendas:", e)
    raise
finally:
    if "engine" in locals():
        engine.dispose()

Conexão com PostgreSQL estabelecida!
Dados carregados com sucesso!


In [2]:
# 2. Visualizar os dados de vendas carregados do PostgreSQL

display(df_vendas)

,id_pedido,categoria,quantidade,preco_unitario
0,2,Livros,1,76.13
1,2,Livros,1,85.56
2,3,Acessórios,1,337.66
3,4,Eletrônicos,1,218.69
4,4,Eletrônicos,1,86.47
...,...,...,...,...
16954,13128,Eletrônicos,1,73.44
16955,13130,Roupas,1,297.15
16956,13130,Casa,1,52.66
16957,13130,Casa,1,629.96


In [3]:
# 3. Calcular o faturamento de cada item (quantidade × preço unitário)

df_vendas['faturamento'] = df_vendas['quantidade'] * df_vendas['preco_unitario']
 
display(df_vendas)

,id_pedido,categoria,quantidade,preco_unitario,faturamento
0,2,Livros,1,76.13,76.13
1,2,Livros,1,85.56,85.56
2,3,Acessórios,1,337.66,337.66
3,4,Eletrônicos,1,218.69,218.69
4,4,Eletrônicos,1,86.47,86.47
...,...,...,...,...,...
16954,13128,Eletrônicos,1,73.44,73.44
16955,13130,Roupas,1,297.15,297.15
16956,13130,Casa,1,52.66,52.66
16957,13130,Casa,1,629.96,629.96


In [4]:
# 4. Calcular o faturamento total, os pedidos pagos e o ticket médio
faturamento_total = df_vendas["faturamento"].sum()
total_pedidos = df_vendas["id_pedido"].nunique()
ticket_medio = faturamento_total / total_pedidos if total_pedidos else 0

# Formatar os valores monetários no padrão brasileiro
def formatar_reais(valor):
    return f"R$ {valor:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")

print(f"Faturamento total: {formatar_reais(faturamento_total)}")
print(f"Pedidos pagos: {total_pedidos}")
print(f"Ticket médio: {formatar_reais(ticket_medio)}")

Faturamento total: R$ 5.711.887,26
Pedidos pagos: 10479
Ticket médio: R$ 545,08


In [5]:
# 5. Analisar faturamento, pedidos distintos e ticket médio por categoria
df_categoria = df_vendas.groupby("categoria").agg(
    faturamento=("faturamento", "sum"),
    quantidade_pedidos=("id_pedido", "nunique")
).reset_index()

df_categoria["ticket_medio"] = (
    df_categoria["faturamento"]
    / df_categoria["quantidade_pedidos"]
).round(2)

display(df_categoria)

,categoria,faturamento,quantidade_pedidos,ticket_medio
0,Acessórios,581310.81,2433,238.93
1,Casa,705365.13,2392,294.89
2,Eletrônicos,3050341.47,2602,1172.31
3,Livros,342292.99,2278,150.26
4,Roupas,1032576.86,3641,283.60
